# Study 1, Amendment 2: CheXzero vs CLEAR on VinBigData (VinDr-CXR source), radiologist labels

Analysis fixed in the OSF amendment dated 3 October 2026 (Amendment 2). Uses the VinBigData competition training images under the competition rules (non-commercial academic research). **No images are saved or exported by this notebook.**

**Settings:** Accelerator **GPU T4 x2**, Internet **On**. Inputs: the competition **VinBigData Chest X-ray Abnormalities Detection** (Add Input, Competitions) and your private **chexzero-weights** dataset.
**Run:** Save Version, then Save & Run All (Commit). About 2 hours. At the end copy the block between `===== VINBIGDATA SUMMARY =====` and `===== END =====`.

In [ ]:
# 0. Settings
import os, sys, time, json, glob, subprocess, hashlib, platform
os.environ["HF_HOME"] = "/tmp/hf"          # large downloads go to /tmp, not the 20 GB output folder
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
SEED = 2026
OUT = "/kaggle/working/vin_out"; os.makedirs(OUT, exist_ok=True)
FINDINGS_NIH = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
PROMPT_WORDS = {"Atelectasis":"atelectasis","Cardiomegaly":"cardiomegaly","Effusion":"pleural effusion",
                "Infiltration":"infiltration","Mass":"mass","Nodule":"nodule","Pneumonia":"pneumonia",
                "Pneumothorax":"pneumothorax","Consolidation":"consolidation","Edema":"edema",
                "Emphysema":"emphysema","Fibrosis":"fibrosis","Pleural_Thickening":"pleural thickening",
                "Hernia":"hernia"}
run_info = {"started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "seed": SEED,
            }
T0 = time.time()

In [ ]:
# 1. Packages and versions
!pip install -q git+https://github.com/openai/CLIP.git gdown
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!pip install -q /kaggle/working/CLEAR
import torch, numpy as np, pandas as pd, transformers
run_info["clear_git_commit"] = subprocess.run(["git","-C","/kaggle/working/CLEAR","rev-parse","HEAD"],
                                              capture_output=True, text=True).stdout.strip()
try:
    from huggingface_hub import HfApi
    run_info["clear_hf_sha"] = HfApi().model_info("peterhan91/CLEAR").sha
except Exception as e:
    run_info["clear_hf_sha"] = "unavailable: " + repr(e)[:200]
run_info.update({"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__,
                 "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
print(json.dumps(run_info, indent=1))
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"
!pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
print("pydicom", pydicom.__version__)

In [ ]:
# 2. VinBigData cohort: majority (2 of 3) and any-radiologist labels; sex and age from DICOM headers
from concurrent.futures import ThreadPoolExecutor
vin_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.csv" in files and "train" in dirs:
        vin_dir = root; break
assert vin_dir, "VinBigData competition data not found: add it under Input (Competitions)"
lab = pd.read_csv(os.path.join(vin_dir, "train.csv"))
print("train.csv rows:", len(lab), "| columns:", list(lab.columns))
VIN = {"Aortic enlargement": "aortic enlargement", "Atelectasis": "atelectasis", "Calcification": "calcification",
       "Cardiomegaly": "cardiomegaly", "Consolidation": "consolidation", "ILD": "interstitial lung disease",
       "Infiltration": "infiltration", "Lung Opacity": "lung opacity", "Nodule/Mass": "nodule or mass",
       "Pleural effusion": "pleural effusion", "Pleural thickening": "pleural thickening", "Pneumothorax": "pneumothorax",
       "Pulmonary fibrosis": "pulmonary fibrosis"}
FINDINGS = list(VIN); POS = [VIN[f] for f in FINDINGS]; NEG = ["no " + w for w in POS]
assert set(FINDINGS) <= set(lab.class_name.unique()), set(FINDINGS) - set(lab.class_name.unique())
images = sorted(lab.image_id.unique())
nrad = lab[lab.class_name.isin(FINDINGS)].groupby(["image_id", "class_name"]).rad_id.nunique().unstack(fill_value=0)
nrad = nrad.reindex(index=images, columns=FINDINGS, fill_value=0)
cohort = pd.DataFrame({"image_id": images})
cohort["path"] = [os.path.join(vin_dir, "train", i + ".dicom") for i in images]
assert all(os.path.exists(p) for p in cohort.path[:20])
for f in FINDINGS:
    cohort[f] = (nrad[f].values >= 2).astype(int); cohort["any_" + f] = (nrad[f].values >= 1).astype(int)
def header(p):
    d = pydicom.dcmread(p, stop_before_pixels=True)
    sex = str(getattr(d, "PatientSex", "") or "").strip().upper()
    age = str(getattr(d, "PatientAge", "") or "").strip().upper()
    try: a = int(age[:-1]) if age.endswith("Y") else (int(age) if age.isdigit() else None)
    except ValueError: a = None
    if a is not None and (a <= 0 or a > 100): a = None
    return sex if sex in ("M", "F") else None, a
with ThreadPoolExecutor(16) as ex: hdr = list(ex.map(header, cohort.path))
cohort["sex"] = [h[0] for h in hdr]; cohort["age"] = [h[1] for h in hdr]
cohort["age_band4"] = pd.cut(cohort.age, [-1, 18, 40, 60, 200], labels=["0-18", "18-40", "40-60", "60+"]).astype(str)
cohort.loc[cohort.age.isna(), "age_band4"] = None
run_info.update({"prompts_pos": POS, "prompts_neg": NEG, "n_images": len(cohort),
                 "sex_counts": cohort.sex.value_counts(dropna=False).to_dict(),
                 "age_missing": int(cohort.age.isna().sum()),
                 "age_band_counts": cohort.age_band4.value_counts(dropna=False).to_dict(),
                 "positives_majority": {f: int(cohort[f].sum()) for f in FINDINGS},
                 "positives_any": {f: int(cohort["any_" + f].sum()) for f in FINDINGS}})
print(json.dumps({k: run_info[k] for k in ["n_images", "sex_counts", "age_missing", "age_band_counts", "positives_majority"]}, indent=1, default=str))
cohort.drop(columns=["path"]).to_csv(f"{OUT}/cohort.csv", index=False)

In [ ]:
# 4. Prompt embeddings with SFR-Embedding-Mistral, following CLEAR's get_embed.py
#    (AutoModel, fp16, last-token pooling, L2 normalisation). This runs in a separate process so
#    that all its GPU memory is released when it finishes. Each text is embedded on its own (no padding).
#    Two prompt versions are saved: plain text, as in the authors' exp_zeroshot.py (primary), and with
#    the instruction prefix that reproduces the authors' released concept embeddings (secondary).
import gc
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from clear.concepts import ConceptBank
concepts_csv = hf_hub_download("peterhan91/CLEAR", "mimic_concepts.csv")
clip_pt = hf_hub_download("peterhan91/CLEAR", "concept_embeddings_368294.pt")
llm_pkl = hf_hub_download("peterhan91/CLEAR", "concepts_embeddings_sfr_mistral.pickle")
bank = ConceptBank.from_files(concepts_csv, clip_embeddings_path=clip_pt, llm_embeddings_path=llm_pkl,
                              allow_unsafe_pickle=True)
C_llm = bank.ensure_llm_embeddings()
pick = np.random.default_rng(SEED).choice(bank.num_concepts, 40, replace=False)
INSTR = "Instruct: Given medical radiology concepts, retrieve relevant passages and generate embeddings\nQuery: "
json.dump({"pos": POS, "neg": NEG, "check": [bank.concepts[i] for i in pick], "instr": INSTR},
          open("/kaggle/working/sfr_in.json", "w"))
SFR_SCRIPT = r'''
import json, torch, numpy as np, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
d = json.load(open("/kaggle/working/sfr_in.json"))
tok = AutoTokenizer.from_pretrained("Salesforce/SFR-Embedding-Mistral")
mm = {i: "13GiB" for i in range(torch.cuda.device_count())}; mm["cpu"] = "24GiB"
llm = AutoModel.from_pretrained("Salesforce/SFR-Embedding-Mistral", dtype=torch.float16,
                                device_map="auto", max_memory=mm).eval()
dev0 = next(llm.parameters()).device
def pool(h, m):
    if m[:, -1].sum() == m.shape[0]:
        return h[:, -1]
    return h[torch.arange(h.shape[0], device=h.device), m.sum(1) - 1]
@torch.no_grad()
def emb(texts):
    out = []
    for s in texts:
        e = tok(s, max_length=8192, truncation=True, return_tensors="pt").to(dev0)
        out.append(F.normalize(pool(llm(**e).last_hidden_state, e["attention_mask"]).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
I = d["instr"]
np.savez("/kaggle/working/sfr_out.npz",
         pos=emb(d["pos"]), neg=emb(d["neg"]),
         pos_i=emb([I + s for s in d["pos"]]), neg_i=emb([I + s for s in d["neg"]]),
         chk=emb(d["check"]), chk_i=emb([I + s for s in d["check"]]))
print("SFR embeddings done")
'''
open("/kaggle/working/sfr_embed.py", "w").write(SFR_SCRIPT)
t = time.time()
r = subprocess.run([sys.executable, "/kaggle/working/sfr_embed.py"], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-3000:])
assert r.returncode == 0, "SFR embedding step failed"
z = np.load("/kaggle/working/sfr_out.npz")
pos_llm, neg_llm, pos_llm_i, neg_llm_i = z["pos"], z["neg"], z["pos_i"], z["neg_i"]
for k in ["pos", "neg", "pos_i", "neg_i"]:
    np.save(f"{OUT}/prompt_emb_sfr_{k}.npy", z[k])
ref = C_llm[pick] / np.linalg.norm(C_llm[pick], axis=1, keepdims=True)
cos_plain = (z["chk"] * ref).sum(1); cos_instr = (z["chk_i"] * ref).sum(1)
run_info["sfr_check_cos_plain_mean_min"] = [float(cos_plain.mean()), float(cos_plain.min())]
run_info["sfr_check_cos_instruct_mean_min"] = [float(cos_instr.mean()), float(cos_instr.min())]
run_info["n_concepts"] = bank.num_concepts; run_info["llm_dim"] = int(C_llm.shape[1])
print("Reproduction of released concept embeddings, cosine mean/min: plain", run_info["sfr_check_cos_plain_mean_min"],
      "| with instruction", run_info["sfr_check_cos_instruct_mean_min"])
subprocess.run(["rm", "-rf", "/tmp/hf/hub/models--Salesforce--SFR-Embedding-Mistral"])
gc.collect(); torch.cuda.empty_cache()
print("GPU memory in use after step 4 (GB):", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])
run_info["sec_prompt_embedding"] = round(time.time()-t)

In [ ]:
# 5. Load CLEAR and build the concept projection.
#    The authors' pipeline computes  normalise( (img @ norm(C_clip).T) @ C_llm ).
#    Because this is linear, it equals  normalise( img @ M ) with M = norm(C_clip).T @ C_llm,
#    which is computed once. Both forms are compared on real images in step 7.
from clear import load_pretrained, tokenize
dev = "cuda:0"
dev_cz = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
clear_model, clear_pre = load_pretrained(device=dev); clear_model.eval()
C_clip = torch.from_numpy(bank.ensure_clip_embeddings()).float()

# check that the released concept CLIP embeddings match CLEAR's text encoder (500 random concepts)
pick2 = np.random.default_rng(SEED+1).choice(bank.num_concepts, 500, replace=False)
with torch.no_grad():
    enc = F.normalize(clear_model.encode_text(tokenize([bank.concepts[i] for i in pick2]).to(dev)).float(), dim=-1).cpu()
cos_clip = (enc * F.normalize(C_clip[pick2], dim=-1)).sum(1)
run_info["concept_clip_check_cos_mean_min"] = [float(cos_clip.mean()), float(cos_clip.min())]
print("Released concept CLIP embeddings vs text encoder, cosine mean/min:", run_info["concept_clip_check_cos_mean_min"])
if cos_clip.min() < 0.999:
    print("Mismatch: recomputing all concept CLIP embeddings with the text encoder")
    chunks = []
    with torch.no_grad():
        for i in range(0, bank.num_concepts, 2048):
            chunks.append(clear_model.encode_text(tokenize(list(bank.concepts[i:i+2048])).to(dev)).float().cpu())
    C_clip = torch.cat(chunks); run_info["concept_clip_recomputed"] = True
else:
    run_info["concept_clip_recomputed"] = False

M = torch.zeros(C_clip.shape[1], C_llm.shape[1], dtype=torch.float64, device=dev)
C_llm_t = torch.from_numpy(C_llm)
for i in range(0, C_clip.shape[0], 8192):
    a = F.normalize(C_clip[i:i+8192].to(dev), dim=1).double()
    M += a.T @ C_llm_t[i:i+8192].to(dev).double()
M = M.float()
np.save(f"{OUT}/clear_concept_projection_M.npy", M.cpu().numpy())
pos_t = F.normalize(torch.from_numpy(pos_llm).to(dev), dim=-1); neg_t = F.normalize(torch.from_numpy(neg_llm).to(dev), dim=-1)
pos_ti = F.normalize(torch.from_numpy(pos_llm_i).to(dev), dim=-1); neg_ti = F.normalize(torch.from_numpy(neg_llm_i).to(dev), dim=-1)
with torch.no_grad():
    cpos = F.normalize(clear_model.encode_text(tokenize(POS).to(dev)).float(), dim=-1)
    cneg = F.normalize(clear_model.encode_text(tokenize(NEG).to(dev)).float(), dim=-1)
print("M:", tuple(M.shape))

In [ ]:
# 6. Load the 10 CheXzero checkpoints: from an attached Kaggle dataset if present, otherwise Google Drive
import gdown, clip
from torchvision.transforms import Compose, Normalize, Resize, InterpolationMode
def find_pts():
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        hits += [os.path.join(root, f) for f in files if f.endswith(".pt")]
    return sorted(hits)
ckpts = find_pts()
if len(ckpts) < 10:
    os.makedirs("/kaggle/working/chexzero_weights", exist_ok=True)
    gdown.download_folder("https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
                          output="/kaggle/working/chexzero_weights", quiet=True, remaining_ok=True)
    ckpts = sorted(glob.glob("/kaggle/working/chexzero_weights/**/*.pt", recursive=True))
assert len(ckpts) == 10, f"expected 10 CheXzero checkpoints, found {len(ckpts)}: {ckpts}"
print("CheXzero checkpoints from:", os.path.dirname(ckpts[0]))
run_info["chexzero_checkpoints"] = [[os.path.basename(c), os.path.getsize(c)] for c in ckpts]
cz_models, cz_text = [], []
for c in ckpts:
    m, _ = clip.load("ViT-B/32", device="cpu", jit=False)
    m.load_state_dict(torch.load(c, map_location="cpu")); m = m.to(dev_cz).eval()
    with torch.no_grad():
        tp = F.normalize(m.encode_text(clip.tokenize(POS, context_length=77).to(dev_cz)).float(), dim=-1)
        tn = F.normalize(m.encode_text(clip.tokenize(NEG, context_length=77).to(dev_cz)).float(), dim=-1)
    cz_models.append(m); cz_text.append((tp, tn))
print("CheXzero checkpoints loaded:", len(cz_models), "| GPU GB in use:", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])

In [ ]:
# 7. One pass over all images: CheXzero ensemble (padded 320), CLEAR padded 448 (concept plain, concept instructed, direct)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
def read_xray(p):
    d = pydicom.dcmread(p); a = d.pixel_array
    try: a = apply_voi_lut(a, d)
    except Exception: pass
    a = a.astype(np.float32)
    if getattr(d, "PhotometricInterpretation", "") == "MONOCHROME1": a = a.max() - a
    a -= a.min(); a /= max(float(a.max()), 1e-6)
    return Image.fromarray((a * 255).astype(np.uint8))
def pad_square(img, size):
    w, h = img.size; r = float(size) / max(w, h)
    img = img.resize((max(1, int(w * r)), max(1, int(h * r))), Image.LANCZOS)
    out = Image.new("L", (size, size)); out.paste(img, ((size - img.size[0]) // 2, (size - img.size[1]) // 2))
    return out
cz_tf = Compose([Normalize((101.48761,)*3, (83.43944,)*3), Resize(224, interpolation=InterpolationMode.BICUBIC)])
cl_norm = Normalize((101.48761/255,)*3, (83.43944/255,)*3)
class DS(Dataset):
    def __init__(self, paths): self.p = list(paths)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try:
            im = read_xray(self.p[i]); ok = 1
        except Exception:
            im = Image.new("L", (448, 448)); ok = 0
        g = np.asarray(pad_square(im, 320), dtype=np.float32)
        return cz_tf(torch.from_numpy(np.repeat(g[None], 3, axis=0))), cl_norm(ToTensor()(pad_square(im, 448).convert("RGB"))), ok
BS = 32
dl = DataLoader(DS(cohort.path), batch_size=BS, num_workers=4, pin_memory=True)
def pair_prob(f, tp, tn):
    p, n = f @ tp.T, f @ tn.T
    return torch.exp(p) / (torch.exp(p) + torch.exp(n))
R = {k: [] for k in ["chexzero", "clear_concept", "clear_concept_instr", "clear_direct"]}; OK = []
t = time.time()
with torch.no_grad():
    for b, (x_cz, x_pad, ok) in enumerate(dl):
        x_cz = x_cz.to(dev_cz, non_blocking=True)
        per = [pair_prob(F.normalize(m.encode_image(x_cz).float(), dim=-1), tp, tn) for m, (tp, tn) in zip(cz_models, cz_text)]
        R["chexzero"].append(torch.stack(per, 0).mean(0).cpu().numpy())
        fi = F.normalize(clear_model.encode_image(x_pad.to(dev)).float(), dim=-1); rep_ = F.normalize(fi @ M, dim=-1)
        R["clear_concept"].append(pair_prob(rep_, pos_t, neg_t).cpu().numpy())
        R["clear_concept_instr"].append(pair_prob(rep_, pos_ti, neg_ti).cpu().numpy())
        R["clear_direct"].append(pair_prob(fi, cpos, cneg).cpu().numpy())
        OK.append(ok.numpy())
        if b % 50 == 0: print(f"{min((b+1)*BS, len(cohort))}/{len(cohort)} images, {round((time.time()-t)/60,1)} min", flush=True)
S = {k: np.concatenate(v).astype(np.float64) for k, v in R.items()}; OK = np.concatenate(OK).astype(bool)
run_info["unreadable_images"] = int((~OK).sum()); run_info["min_scoring"] = round((time.time()-t)/60, 1)
for k, v in S.items(): np.save(f"{OUT}/scores_{k}.npy", v.astype(np.float32))
print("unreadable:", run_info["unreadable_images"])

In [ ]:
# 8. Analysis (decision rule in Amendment 2): image-level bootstrap, 1,000 resamples, seed 2026
keep = OK
C = cohort[keep].reset_index(drop=True); SS = {k: v[keep] for k, v in S.items()}
n = len(C); pidx = np.arange(n)
rng = np.random.default_rng(SEED)
WB = [np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64) for _ in range(1000)]
class Cell:
    def __init__(self, s, y, p):
        o = np.argsort(s, kind="mergesort"); s = s[o]; self.y, self.p = y[o].astype(np.float64), p[o]
        self.starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    def auc(self, w=None):
        wi = 1.0 if w is None else w[self.p]
        bp = np.add.reduceat(wi * self.y, self.starts); bn = np.add.reduceat(wi * (1 - self.y), self.starts)
        P, N = bp.sum(), bn.sum()
        return np.nan if P == 0 or N == 0 else float((bp * (np.cumsum(bn) - bn + 0.5 * bn)).sum() / (P * N))
def ci(a, lvl=0.95):
    a = a[~np.isnan(a)]
    return (float("nan"), float("nan")) if a.size == 0 else (float(np.quantile(a, (1-lvl)/2)), float(np.quantile(a, 1-(1-lvl)/2)))
def table(s, Y, mask):
    idx = np.flatnonzero(mask); pt, bt = [], []
    for j in range(len(FINDINGS)):
        c = Cell(s[idx, j], Y[idx, j], pidx[idx]); pt.append(c.auc()); bt.append([c.auc(w) for w in WB])
    return np.array(pt), np.array(bt).T          # (findings,), (B, findings)
def eligible_overall(Y): return [j for j in range(len(FINDINGS)) if Y[:, j].sum() >= 30]
OUTR = {}
for labname, Y in [("majority", C[FINDINGS].values), ("any", C[["any_" + f for f in FINDINGS]].values)]:
    E = eligible_overall(Y); res = {"eligible": [FINDINGS[j] for j in E]}
    tabs = {k: table(SS[k], Y, np.ones(n, bool)) for k in SS}
    pc, bc = tabs["chexzero"]
    for k, (pk, bk) in tabs.items():
        r = {"macro": float(np.nanmean(pk[E])), "macro_ci95": ci(np.nanmean(bk[:, E], axis=1)), "per_finding": {FINDINGS[j]: round(float(pk[j]), 3) for j in E}}
        if k != "chexzero":
            r["delta_vs_chexzero"] = float(np.nanmean(pk[E]) - np.nanmean(pc[E])); r["delta_ci95"] = ci(np.nanmean(bk[:, E], axis=1) - np.nanmean(bc[:, E], axis=1))
            r["wins_losses"] = [int((pk[E] > pc[E]).sum()), int((pk[E] < pc[E]).sum())]
        res[k] = r
    OUTR[labname] = res
p = OUTR["majority"]["clear_concept"]; lo, hi = p["delta_ci95"]; wl = p["wins_losses"]
verdict = ("CONSISTENT with the CLEAR paper (CLEAR ahead of CheXzero)" if p["delta_vs_chexzero"] > 0 and wl[0] > wl[1]
           else "INCONSISTENT with the CLEAR paper (CLEAR behind CheXzero, interval excludes 0)" if p["delta_vs_chexzero"] < 0 and hi < 0
           else "INCONCLUSIVE")
# subgroup replication (majority labels)
Y = C[FINDINGS].values
grp = {"sex_F": (C.sex == "F").values, "sex_M": (C.sex == "M").values}
for a in ["0-18", "18-40", "40-60", "60+"]: grp[f"age_{a}"] = (C.age_band4 == a).values
def elig(groups): return [j for j in range(len(FINDINGS)) if all(Y[grp[g], j].sum() >= 30 for g in groups)]
E_SEX = elig(["sex_F", "sex_M"]); E_AGE = elig([f"age_{a}" for a in ["0-18", "18-40", "40-60", "60+"]])
if len(E_AGE) == 0:   # 0-18 may be too small in VinDr; fall back to adult bands only and report it
    E_AGE = elig(["age_18-40", "age_40-60", "age_60+"]); AGE_CONTRASTS = ["18-40", "60+"]
else:
    AGE_CONTRASTS = ["0-18", "18-40", "60+"]
SUB = {"sex_eligible": [FINDINGS[j] for j in E_SEX], "age_eligible": [FINDINGS[j] for j in E_AGE], "age_contrasts": AGE_CONTRASTS,
       "n_by_group": {g: int(m.sum()) for g, m in grp.items()}}
for k in ["chexzero", "clear_concept", "clear_direct"]:
    tabs = {g: table(SS[k], Y, m) for g, m in grp.items()}
    def mac(g, E): pt, bt = tabs[g]; return float(np.nanmean(pt[E])), np.nanmean(bt[:, E], axis=1)
    r = {}
    if E_SEX:
        pF, bF = mac("sex_F", E_SEX); pM, bM = mac("sex_M", E_SEX); r["sex_F_minus_M"] = (pF - pM, ci(bF - bM, .95))
    if E_AGE:
        pR, bR = mac("age_40-60", E_AGE)
        for a in AGE_CONTRASTS:
            pa, ba = mac(f"age_{a}", E_AGE); r[f"age_{a}_minus_40-60"] = (pa - pR, ci(ba - bR, 1 - 0.05/3))
    SUB[k] = r
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(x) for x in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
json.dump(jsonable({"run_info": run_info, "accuracy": OUTR, "verdict": verdict, "subgroups": SUB}), open(f"{OUT}/vin_results.json", "w"), indent=1, default=str)
!cd /kaggle/working && rm -f vin_outputs.zip && zip -qr vin_outputs.zip vin_out
print("===== VINBIGDATA SUMMARY =====")
print("images", n, "| unreadable", run_info["unreadable_images"], "| sex", run_info["sex_counts"], "| age missing", run_info["age_missing"])
print("age bands", run_info["age_band_counts"])
print("SFR check plain/instr:", run_info.get("sfr_check_cos_plain_mean_min"), run_info.get("sfr_check_cos_instruct_mean_min"))
for labname in ["majority", "any"]:
    res = OUTR[labname]; print(f"-- {labname} labels, eligible findings: {len(res['eligible'])}")
    for k in SS:
        r = res[k]; line = f"  {k:<20} macro {r['macro']:.3f} ({r['macro_ci95'][0]:.3f}-{r['macro_ci95'][1]:.3f})"
        if "delta_vs_chexzero" in r: line += f" | minus CheXzero {r['delta_vs_chexzero']:+.3f} ({r['delta_ci95'][0]:+.3f} to {r['delta_ci95'][1]:+.3f}) | wins/losses {r['wins_losses']}"
        print(line)
print("-- per finding (majority): chexzero / clear_concept / clear_direct")
for f in OUTR["majority"]["eligible"]:
    print(f"  {f:<20} {OUTR['majority']['chexzero']['per_finding'][f]:.3f} / {OUTR['majority']['clear_concept']['per_finding'][f]:.3f} / {OUTR['majority']['clear_direct']['per_finding'][f]:.3f}")
print("VERDICT:", verdict)
print("-- subgroups (majority labels):", {k: SUB[k] for k in ["sex_eligible", "age_eligible", "age_contrasts", "n_by_group"]})
for k in ["chexzero", "clear_concept", "clear_direct"]:
    for g, (p_, (lo_, hi_)) in SUB[k].items(): print(f"  {k:<14} {g:<22} {p_:+.3f} ({lo_:+.3f} to {hi_:+.3f})")
print("minutes:", round((time.time()-T0)/60, 1))
print("===== END =====")